# Μείωση Πλεονασμού Αισθητήρων Γραμμής Παραγωγής με την PROC GVARCLUS

## Περίληψη

Μια γραμμή παραγωγής πολλαπλών ζωνών μεταδίδει δεκάδες συσχετισμένες μετρήσεις αισθητήρων, πολλές από τις οποίες φέρουν το ίδιο υποκείμενο σήμα. Αυτό το notebook χρησιμοποιεί την **PROC GVARCLUS** (ομαδοποίηση μεταβλητών) για να ομαδοποιήσει τους 13 αισθητήρες διεργασίας σε τέσσερις ξένες μεταξύ τους ομάδες, και στη συνέχεια διαβάζει τις τιμές R-τετράγωνο κάθε ομάδας για να επιλέξει ένα αντιπροσωπευτικό όργανο μέτρησης ανά ομάδα — μειώνοντας το αποτύπωμα παρακολούθησης SPC χωρίς απώλεια πληροφορίας διεργασίας. Τρεις από τις τέσσερις ομάδες αντιστοιχούν καθαρά σε φυσικές ζώνες (μέσο R-τετράγωνο 0,92, 0,93 και 0,96)· η τέταρτη είναι μια ομάδα ενός μόνο καναλιού που η διαδικασία απομόνωσε ξεχωριστά, την οποία εξετάζουμε αντί να την αγνοήσουμε.

## Πηγές Δεδομένων

Όλα τα δεδομένα παράγονται εσωτερικά με την `call streaminit(20260531)` και την `rand()` — δεν υπάρχουν εξωτερικές ή δικτυακές εισόδους.

| Σύνολο Δεδομένων | Γραμμές | Βασικές Μεταβλητές | Περιγραφή |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Συνθετικές μετρήσεις από μια γραμμή παραγωγής 3 ζωνών. Οι αισθητήρες εντός μιας ζώνης μοιράζονται έναν λανθάνοντα οδηγό (υψηλή συσχέτιση)· όργανα διαφορετικών ζωνών (θερμοκρασίες συνδεδεμένες με τη ζώνη 1, πιέσεις με τη ζώνη 2, δόνηση με τη ζώνη 3) προστίθενται για να δημιουργήσουν ρεαλιστικό πλεονασμό. Ο βρόχος του βήματος DATA ζητά 300 κύκλους, αλλά αυτό το περιβάλλον αξιολόγησης εκτελείται χωρίς άδεια χρήσης και διατηρεί τις πρώτες 100 παρατηρήσεις — αρκετές για να ανακτηθεί καθαρά η δομή ομαδοποίησης. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | Μία γραμμή ανά αισθητήρα εισόδου: η ομάδα στην οποία ανατέθηκε και το R-τετράγωνό του με το στοιχείο εκείνης της ομάδας. Παράγεται από τη δήλωση `OUTPUT OUT=`. |

# Μείωση Πλεονασμού Αισθητήρων Γραμμής Παραγωγής με την PROC GVARCLUS

Σε μια οργανομετρημένη γραμμή παραγωγής, είναι σύνηθες να καταγράφονται δεκάδες αισθητήρες που μετρούν επικαλυπτόμενα φυσικά φαινόμενα — πολλαπλά θερμοζεύγη σε μία ζώνη, πλεονάζοντες μεταλλάκτες πίεσης, αισθητήρες δόνησης που παρακολουθούν τον ίδιο άξονα. Η τροφοδότηση κάθε καναλιού σε ένα διάγραμμα ελέγχου ή σε ένα μοντέλο επόμενου σταδίου σπαταλά προσπάθεια παρακολούθησης και αυξάνει την πολυσυγγραμμικότητα.

Η **ομαδοποίηση μεταβλητών** αντιμετωπίζει αυτό απευθείας. Η `PROC GVARCLUS` διαμερίζει τις αριθμητικές μεταβλητές σε *ξένες μεταξύ τους* ομάδες, όπου κάθε ομάδα συνοψίζεται από την πρώτη κύρια συνιστώσα των μελών της. Αισθητήρες που κινούνται μαζί καταλήγουν στην ίδια ομάδα· ο μηχανικός μπορεί στη συνέχεια να διατηρήσει ένα αντιπροσωπευτικό κανάλι ανά ομάδα.

Σε αυτό το notebook:

1. Συνθέτουμε μετρήσεις από μια γραμμή 3 ζωνών με σκόπιμα πλεονάζοντες αισθητήρες (ο βρόχος ζητά 300 κύκλους· 100 διατηρούνται εδώ).
2. Ομαδοποιούμε τα 13 κανάλια με την `PROC GVARCLUS` στο `MAXCLUSTERS=4`.
3. Καταγράφουμε τις αναθέσεις ομάδων σε ένα σύνολο δεδομένων εξόδου και τις συνοψίζουμε.
4. Ερμηνεύουμε τις τιμές R-τετράγωνο για να επιλέξουμε ένα όργανο μέτρησης ανά ομάδα για συνεχή SPC.

## Βήμα 1 — Παραγωγή συνθετικών δεδομένων αισθητήρων

Προσομοιώνουμε τρεις ζώνες διεργασίας, καθεμία με έναν κρυφό οδηγό (`zone1_a`, `zone2_a`, `zone3_a`). Τα υπόλοιπα κανάλια είναι θορυβώδεις γραμμικές συναρτήσεις του οδηγού της ζώνης τους, οπότε είναι έντονα συσχετισμένα εντός μιας ζώνης αλλά σχεδόν ανεξάρτητα μεταξύ ζωνών. Επίσης συνδέουμε τις θερμοκρασίες εισόδου/εξόδου με τη ζώνη 1 και τους δύο μεταλλάκτες πίεσης με τη ζώνη 2, μιμούμενοι τον πλεονασμό μεταξύ οργάνων που παρατηρείται σε πραγματικές γραμμές. Ένας σταθερός σπόρος καθιστά την εκτέλεση αναπαραγώγιμη. Ο βρόχος ζητά 300 κύκλους· σε λειτουργία χωρίς άδεια χρήσης η μηχανή διατηρεί τις πρώτες 100 παρατηρήσεις, όπως αναφέρει η παρακάτω σημείωση NOTE.

In [1]:
ΔΕΔΟΜΕΝΑ process_sensors;
    CALL streaminit(20260531);
    ΕΠΑΝΑΛΗΨΗ cycle = 1 ΕΩΣ 300;
        /* Zone 1: latent driver plus two redundant probes */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zone 2: latent driver plus two redundant probes */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zone 3: latent driver plus one redundant probe */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Cross-instrument channels tied to existing drivers */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        ΕΞΟΔΟΣ;
    ΤΕΛΟΣ;
    ΑΦΑΙΡΕΣΗ cycle;
ΕΚΤΕΛΕΣΗ;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds


## Βήμα 2 — Ομαδοποίηση των αισθητήρων

Καλούμε την `PROC GVARCLUS` σε όλα τα 13 κανάλια. Η δήλωση `VAR` απαριθμεί τους αριθμητικούς αισθητήρες προς ομαδοποίηση. Περιορίζουμε τη διαμέριση σε τέσσερις ομάδες με `MAXCLUSTERS=4` (αναμένουμε περίπου μία ομάδα ανά φυσική ζώνη, με λίγο περιθώριο). Η `ODS GRAPHICS ON` με `PLOTS=ALL` ενεργοποιεί το δενδρόγραμμα ομαδοποίησης μεταβλητών· η μηχανή γράφει αυτό το SVG στον φάκελο εργασίας αντί να το αποδίδει ενσωματωμένο, οπότε η δομή που διαβάζουμε παρακάτω προέρχεται από τον τυπωμένο πίνακα **Αναθέσεις Ομάδων Μεταβλητών** και τον πίνακα ιδιοτιμών ανά ομάδα.

Η δήλωση `OUTPUT OUT=` γράφει τις αναθέσεις μεταβλητής-σε-ομάδα — μαζί με το R-τετράγωνο κάθε μεταβλητής έναντι της δικής της ομάδας — σε ένα σύνολο δεδομένων στο οποίο μπορούμε να προγραμματίσουμε αργότερα.

In [2]:
ODS GRAPHICS ON;

ΔΙΑΔΙΚΑΣΙΑ gvarclus ΔΕΔΟΜΕΝΑ=process_sensors maxclusters=4 PLOTS=ALL;
    ΜΕΤΑΒΛΗΤΗ zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    ΕΞΟΔΟΣ out=clusters;
ΕΚΤΕΛΕΣΗ;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Βήμα 3 — Επανεκτέλεση με την επιλογή SUMMARY

Η εκτέλεση της διαδικασίας μια δεύτερη φορά με την επιλογή `SUMMARY` διατηρεί την ίδια διαμέριση. Η `PLOTS=NONE` απενεργοποιεί τα γραφικά σε αυτό το πέρασμα. Σε αυτό το περιβάλλον η τυπωμένη αναφορά είναι πανομοιότυπη με το Βήμα 2 — ο ίδιος πίνακας ανάθεσης 13 γραμμών, οι ίδιες τέσσερις ομάδες, και η ίδια σύνοψη ιδιοτιμών — οπότε αυτό το κελί κυρίως αποδεικνύει ότι οι επιλογές `SUMMARY` και `PLOTS=NONE` αναλύονται και εκτελούνται σωστά· δεν αναμένεται να προσθέσει νέους αριθμούς.

In [3]:
ΔΙΑΔΙΚΑΣΙΑ gvarclus ΔΕΔΟΜΕΝΑ=process_sensors maxclusters=4 summary PLOTS=none;
    ΜΕΤΑΒΛΗΤΗ zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
ΕΚΤΕΛΕΣΗ;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Βήμα 4 — Εξέταση του συνόλου δεδομένων εξόδου

Το σύνολο δεδομένων `clusters` από την `OUTPUT OUT=` έχει μία γραμμή ανά αισθητήρα με την ανατεθειμένη `Cluster` και το `RSq_Own` (τη συσχέτιση στο τετράγωνο μεταξύ του αισθητήρα και του στοιχείου της ομάδας του). Ένα υψηλό `RSq_Own` σημαίνει ότι ο αισθητήρας αντιπροσωπεύεται καλά από την ομάδα του — ένας ιδανικός υποψήφιος για *αφαίρεση* υπέρ του αντιπροσωπευτικού στοιχείου της ομάδας. Ταξινομούμε κατά ομάδα, και έπειτα κατά R-τετράγωνο, ώστε ο πιο αντιπροσωπευτικός αισθητήρας κάθε ομάδας να βρίσκεται στην κορυφή της ομάδας του.

In [4]:
ΔΙΑΔΙΚΑΣΙΑ ΤΑΞΙΝΟΜΗΣΗ ΔΕΔΟΜΕΝΑ=clusters out=clusters_ranked;
    ΚΑΤΑ CLUSTER DESCENDING RSq_Own;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=clusters_ranked ΕΤΙΚΕΤΑ;
    ΜΕΤΑΒΛΗΤΗ Variable CLUSTER RSq_Own;
    ΕΤΙΚΕΤΑ Variable = "Κανάλι Αισθητήρα"
          CLUSTER  = "Ομάδα"
          RSq_Own  = "R-Τετράγωνο με Δική της Ομάδα";
ΕΚΤΕΛΕΣΗ;


  Obs                 Κανάλι Αισθητήρα       Ομάδα                         R-Τετράγωνο με Δική της Ομάδα
-----  -------------------------------  ----------  ----------------------------------------------------
    1  ZONE1_A                                   1                                               0.96867
    2  ZONE1_B                                   1                                                0.9189
    3  TEMP_IN                                   1                                              0.903394
    4  TEMP_OUT                                  1                                              0.889519
    5  ZONE2_A                                   2                                               0.98364
    6  ZONE2_B                                   2                                              0.946708
    7  PRESSURE_A                                2                                              0.929356
    8  PRESSURE_B                                2    


NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Ερμηνεία των αποτελεσμάτων

Η διαμέριση ανακτά το μεγαλύτερο μέρος της φυσικής δομής της γραμμής, με μία ειλικρινή επιφύλαξη:

- **Η Ομάδα 1** συγκεντρώνει το `zone1_a` (R²=0,969), το `zone1_b` (0,919) και τις θερμοκρασίες εισόδου/εξόδου `temp_in` (0,903) και `temp_out` (0,890) — τέσσερα από τα πέντε κανάλια που οδηγούνται από το λανθάνον σήμα της ζώνης 1. Μέσο R-τετράγωνο 0,920.
- **Η Ομάδα 2** συγκεντρώνει και τα τρία κανάλια της ζώνης 2 `zone2_a` (0,984), `zone2_b` (0,947), `zone2_c` (0,892) μαζί με τους δύο μεταλλάκτες πίεσης `pressure_a` (0,929) και `pressure_b` (0,921), που συνδέσαμε με τον οδηγό της ζώνης 2. Μέσο R-τετράγωνο 0,935.
- **Η Ομάδα 3** συγκεντρώνει το `zone3_a` (0,977), το `zone3_b` (0,949) και το `vibration` (0,959). Μέσο R-τετράγωνο 0,962.
- **Η Ομάδα 4** είναι ένα μόνο κανάλι: το `zone1_c`, ο τρίτος ανιχνευτής της ζώνης 1, απομονώθηκε μόνος του με R²=1,000 (ένα μοναδιαίο στοιχείο εξηγείται, τετριμμένα, τέλεια από τον εαυτό του). Παρόλο που μοιράζεται τον οδηγό της ζώνης 1, σε αυτό το μέγεθος δείγματος η διαδικασία έκρινε το `zone1_c` αρκετά διακριτό από την ομάδα `zone1_a`/θερμοκρασιών ώστε να δικαιολογείται η δική του ξεχωριστή ομάδα αντί να συγχωνευτεί στην Ομάδα 1.

Οι τρεις ομάδες πολλαπλών καναλιών φέρουν η καθεμία μέσο R-τετράγωνο πάνω από **0,90**, επιβεβαιώνοντας ότι μία μόνο συνιστώσα εξηγεί το μεγαλύτερο μέρος της μεταβλητότητας μεταξύ των μελών τους — ακριβώς τον πλεονασμό που θέλουμε να συμπτύξουμε. Η μεμονωμένη ανωμαλία — το `zone1_c` που καταλήγει στη δική του ομάδα αντί με την υπόλοιπη ζώνη 1 — είναι μια χρήσιμη υπενθύμιση ότι η ομαδοποίηση μεταβλητών είναι βασισμένη σε δεδομένα: με περισσότερους κύκλους ή υψηλότερο όριο `MAXCLUSTERS` το όριο μπορεί να μετατοπιστεί, οπότε η διαμέριση είναι ένα σημείο εκκίνησης για τη μηχανική κρίση, όχι μια σταθερή αλήθεια.

**Ενέργεια για τη γραμμή.** Εντός κάθε ομάδας πολλαπλών καναλιών, διατηρήστε τον αισθητήρα με το υψηλότερο `RSq_Own` (το κανάλι που αντιπροσωπεύει καλύτερα την ομάδα του) και αποσύρετε ή υποβαθμίστε σε προτεραιότητα τους υπόλοιπους από τη συνήθη χαρτογράφηση SPC — για παράδειγμα το `zone2_a` για την Ομάδα 2 και το `zone3_a` για την Ομάδα 3. Αντιμετωπίστε το μοναδιαίο στοιχείο `zone1_c` ως σημαία προς διερεύνηση παρά ως αυτόματη διατήρηση: επιβεβαιώστε αν φέρει πράγματι διακριτή πληροφορία ή είναι τεχνούργημα ομαδοποίησης πριν αποφασίσετε να το παρακολουθείτε ξεχωριστά. Ακόμη και με αυτό το ένα κανάλι στην άκρη, αυτό συμπτύσσει 13 παρακολουθούμενα κανάλια σε ένα πλάνο παρακολούθησης τεσσάρων καναλιών, μειώνοντας τον θόρυβο συναγερμών και την πολυσυγγραμμικότητα ενώ διατηρεί το περιεχόμενο πληροφορίας της γραμμής.